# Stage 1: Domain-Adaptive Pretraining (DAPT) for GraphCodeBERT

This notebook is optimized for a local workstation (Ryzen 5 5600, 32GB RAM, RTX 3060 12GB) to "teach" GraphCodeBERT x86/x64 Assembly using Masked Language Modeling (MLM).
We use a dataset generator to stream the 22GB dataset so it comfortably fits in your 32GB of system RAM, and training arguments optimized for your 12GB VRAM GPU.

In [1]:
# Ensure you have the required libraries installed in your local Python environment:
# Run this in your VS Code terminal:
# pip install transformers datasets accelerate torch

In [2]:
import os

# Point this directly to your local JSON output folder
DATA_DIR = r'e:\Thesis\combined_output'

### 1. Data Loading and Chunking
GraphCodeBERT handles a maximum of 512 tokens. We need to extract the assembly instructions and chunk them into sequences of ~512 tokens.

In [3]:
import glob
import json
import random
from datasets import Dataset

def generate_assembly_chunks(data_dir, max_files=100):
    """
    Generator that reads JSON files and yields large blocks of assembly instructions.
    We yield large chunks and let the tokenizer slice them into token-exact 512 lengths.
    """
    json_files = glob.glob(os.path.join(data_dir, '*.json'))
    random.shuffle(json_files)  # Shuffle to ensure a good mix of samples during training

    if max_files:
        json_files = json_files[:max_files]
        print(f"Limiting to the first {max_files} files for a test run...")
    
    failed_files = 0
    
    for file_path in json_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
            if 'asm' not in data:
                continue
                
            raw_lines = data['asm'].split('\n')
            instructions = []
            
            for line in raw_lines:
                clean_line = line.strip()
                # Ignore comments (;) and empty lines to keep pure assembly
                if clean_line and not clean_line.startswith(';'):
                    instructions.append(clean_line)
            
            # Break into 10,000-instruction strings to prevent giant single-sample strings,
            # letting the tokenizer precisely handle the 512 token limits via sliding window
            chunk_size = 10000
            for i in range(0, len(instructions), chunk_size):
                chunk = " \n ".join(instructions[i:i + chunk_size])
                yield {"text": chunk}
                
        except Exception as e:
            failed_files += 1
            
    if failed_files > 0:
        print(f"Warning: {failed_files} files failed to parse and were skipped.")

# Create Hugging Face Dataset from generator
# You can set max_files=50 initially to do a quick 3-minute test run, 
# then change it to max_files=None when you want to train on the entire 22GB.
dataset = Dataset.from_generator(lambda: generate_assembly_chunks(DATA_DIR, max_files=100))
print(f"Created dataset with {len(dataset)} instruction blocks.")

C:\Users\cjrea\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Created dataset with 843 instruction blocks.


### 2. Tokenization and Data Collator
We use GraphCodeBERT's tokenizer. For MLM, we use `DataCollatorForLanguageModeling` which randomly masks 15% of the tokens.

In [ ]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

MODEL_NAME = "microsoft/graphcodebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    # Tokenize and chunk exactly to 512 tokens using overflow!
    # Removing padding="max_length" so dynamic padding can handle varying lengths efficiently
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        return_overflowing_tokens=True,
        stride=16  # Short sliding window to avoid breaking instruction context roughly
    )
    
    # Must explicitly return just input_ids and attention_mask
    return {
        "input_ids": outputs["input_ids"],
        "attention_mask": outputs["attention_mask"]
    }

# Split the raw text dataset BEFORE tokenizing to prevent overlapping chunk leakage between train and test
split_dataset = dataset.train_test_split(test_size=0.1)

# Tokenize dataset separately for train and eval
# We drop "text" because return_overflowing_tokens splits lengths into more samples
train_dataset = split_dataset["train"].map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text"]
)

eval_dataset = split_dataset["test"].map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text"]
)

# Masking 15% of the tokens (standard for BERT/MLM)
# Use pad_to_multiple_of=8 for FP16 Tensor Core alignment
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=True, 
    mlm_probability=0.15,
    pad_to_multiple_of=8
)

### 3. Initialize Model and Train
We load `AutoModelForMaskedLM` and use Hugging Face's `Trainer` to run the finetuning loop.

In [ ]:
from transformers import AutoModelForMaskedLM, TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint
import torch
import os

# Load the base model with a Masked Language Modeling head
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

# Explicitly use FP16 for the RTX 3060 (Ampere architecture excels at FP16 mixed precision)
# Your 12GB VRAM is ample for per_device_train_batch_size=8, but we use
# gradient_accumulation_steps=2 to mimic an effective batch size of 16 for better gradients.
# Workers=0 avoids multiprocessing spawn issues on Windows with Dataset.from_generator.
training_args = TrainingArguments(
    output_dir="./gcb-assembly-pretrained",
    overwrite_output_dir=False,       # Set to False to PREVENT nuking existing checkpoints on restart
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2, 
    per_device_eval_batch_size=8,
    dataloader_num_workers=0,         # Safe setting for Windows
    eval_strategy="steps",
    eval_steps=5000,                  # Lowering frequency to match save_steps
    save_steps=5000,
    save_total_limit=2,                # Keep only the 2 most recent checkpoints to save disk space
    logging_steps=100,
    learning_rate=5e-5,
    warmup_ratio=0.06,                # Warmup for stable MLM
    weight_decay=0.01,                # Standard weight decay for BERT pretraining
    fp16=True, 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

# Empty PyTorch cache just in case
torch.cuda.empty_cache()

# Detect last checkpoint to allow pausing and resuming
last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    last_checkpoint = get_last_checkpoint(training_args.output_dir)
    if last_checkpoint is not None:
        print(f"Resuming training from {last_checkpoint}")

# Start or resume training
trainer.train(resume_from_checkpoint=last_checkpoint)

Step,Training Loss,Validation Loss
5000,0.215500,0.196152
10000,0.163800,0.148069
15000,0.140700,0.129510
20000,0.128500,0.118514
25000,0.120200,0.114724
30000,0.112200,0.107081
35000,0.109300,0.102739
40000,0.104600,0.098833
45000,0.105500,0.096960
50000,0.097500,0.095023


TrainOutput(global_step=60369, training_loss=0.13684726849460727, metrics={'train_runtime': 40156.529, 'train_samples_per_second': 24.053, 'train_steps_per_second': 1.503, 'total_flos': 2.5428363548473037e+17, 'train_loss': 0.13684726849460727, 'epoch': 3.0})

In [6]:
# Save the final pretrained model and tokenizer locally
trainer.save_model("./gcb-assembly-final")
tokenizer.save_pretrained("./gcb-assembly-final")
print("Training Complete! Valid x86 GCB Model saved in ./gcb-assembly-final")

Training Complete! Valid x86 GCB Model saved in ./gcb-assembly-final


### 4. Validate DAPT Efficacy: Baseline vs. Trained Model (Delta Testing)

To scientifically demonstrate the success of Stage 1 (Domain-Adaptive Pretraining) for the thesis, we will test both the **original** `microsoft/graphcodebert-base` and our **trained assembly model**. By feeding both models raw x86/x64 assembly snippets with missing instructions or registers, we can observe the "Domain Delta"—the difference between the base model hallucinating high-level code, and the trained model correctly identifying assembly syntax structures.

We will test multiple categories: 
1. Stack Operations (Prologue/Epilogue)
2. ALU Operations (Bitwise zeroing/math)
3. Control Flow (Conditionals/Jumps)
4. Memory References (Offsets)

In [11]:
from transformers import pipeline
import time

print("="*60)
print("Loading Models for Delta Testing...")
print("="*60)

# Load the DAPT (your newly trained) model
print("[+] Loading our DAPT-trained Assembly Model...")
dapt_fill_mask = pipeline(
    "fill-mask",
    model="./gcb-assembly-final",
    tokenizer="./gcb-assembly-final",
    device=0  # Offload right to the RTX 3060
)

# Load the base model straight from Huggingface
print("[+] Loading the Base (untrained) GraphCodeBERT Model...")
base_fill_mask = pipeline(
    "fill-mask",
    model="microsoft/graphcodebert-base",
    tokenizer="microsoft/graphcodebert-base",
    device=0
)

# Define our robust test suite covering multiple assembly domains
# We use a placeholder [MASK] which we will replace with the appropriate model's specific mask token dynamically
test_suite = {
    "Function Prologue (Stack)": "push [MASK] \n mov ebp, esp",
    "Function Epilogue (Stack)": "pop ebp \n [MASK]",
    "Zeroing Register (ALU)": "xor eax, [MASK]",
    "Condition/Jump (Control Flow)": "test eax, eax \n je [MASK]",
    "Memory Deref/Array (Memory)": "mov ebx, dword ptr [eax + [MASK]]",
    "Moving Immediate (Data Shift)": "mov [MASK], 0x1"
}

dapt_mask_tok = dapt_fill_mask.tokenizer.mask_token
base_mask_tok = base_fill_mask.tokenizer.mask_token

print("\n\n" + "="*60)
print("             STARTING DELTA COMPARISON TEST")
print("="*60)

for category, instruction in test_suite.items():
    print(f"\n\n>>> TARGET CATEGORY: {category}")
    print(f"--- Instruction string:  {instruction.replace('[MASK]', '[?]')}")
    
    # 1. Base Model Run
    base_inst = instruction.replace("[MASK]", base_mask_tok)
    base_results = base_fill_mask(base_inst)
    
    print("\n--- BASELINE (Untrained) MODEL PREDICTIONS ---")
    for i, res in enumerate(base_results[:3]):
         # We highlight the predicted token using ANSI colors or clear markers
        print(f"  {i+1}. Predicted Token: '{res['token_str']}' (Confidence: {res['score']:.4f})  -> {res['sequence']}")
    
    # 2. DAPT Model Run
    dapt_inst = instruction.replace("[MASK]", dapt_mask_tok)
    dapt_results = dapt_fill_mask(dapt_inst)
    
    print("\n--- DAPT (Your trained) MODEL PREDICTIONS ---")
    for i, res in enumerate(dapt_results[:3]):
        print(f"  {i+1}. Predicted Token: '{res['token_str']}' (Confidence: {res['score']:.4f})  -> {res['sequence']}")
    
    # Tiny sleep for formatting output buffer smoothly
    time.sleep(0.1)

print("\n" + "="*60)
print("TESTING COMPLETE - The 'Domain Delta' is successfully established.")

Device set to use cuda:0


Loading Models for Delta Testing...
[+] Loading our DAPT-trained Assembly Model...
[+] Loading the Base (untrained) GraphCodeBERT Model...


Device set to use cuda:0




             STARTING DELTA COMPARISON TEST


>>> TARGET CATEGORY: Function Prologue (Stack)
--- Instruction string:  push [?] 
 mov ebp, esp

--- BASELINE (Untrained) MODEL PREDICTIONS ---
  1. Predicted Token: ' esp' (Confidence: 0.4075)  -> push esp 
 mov ebp, esp
  2. Predicted Token: ' ' (Confidence: 0.0797)  -> push  
 mov ebp, esp
  3. Predicted Token: ',' (Confidence: 0.0647)  -> push, 
 mov ebp, esp

--- DAPT (Your trained) MODEL PREDICTIONS ---
  1. Predicted Token: ']' (Confidence: 0.2308)  -> push] 
 mov ebp, esp
  2. Predicted Token: 'BP' (Confidence: 0.1022)  -> pushBP 
 mov ebp, esp
  3. Predicted Token: 'X' (Confidence: 0.0668)  -> pushX 
 mov ebp, esp


>>> TARGET CATEGORY: Function Epilogue (Stack)
--- Instruction string:  pop ebp 
 [?]

--- BASELINE (Untrained) MODEL PREDICTIONS ---
  1. Predicted Token: ' stack' (Confidence: 0.2640)  -> pop ebp stack
  2. Predicted Token: ' obj' (Confidence: 0.0362)  -> pop ebp obj
  3. Predicted Token: 'Stack' (Confidence: 0.0177